# SiteSafe Vision: AI-Powered Construction PPE Compliance Screening System
### End-to-End Transfer Learning, MLOps Lineage, and Safety-Critical Evaluation

> ⚠️ **SAFETY & REGULATORY DISCLAIMER**: This notebook and the associated models provide an image-based screening aid. This system does NOT perform autonomous safety inspections, guaranteed object detection, or certified workplace safety assessments.

## 1. Setup & Environment Verification

In [ ]:
import sys
import json
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
set_seed(42)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 2. Dataset Provenance & Ingestion Metadata

In [ ]:
meta_path = PROJECT_ROOT / "data" / "raw" / "dataset_metadata.json"
with open(meta_path, "r", encoding="utf-8") as f:
    dataset_metadata = json.load(f)

print(f"Dataset: {dataset_metadata['dataset_name']}")
print(f"Source URL: {dataset_metadata['source_url']}")
print(f"License: {dataset_metadata['license']}")
print(f"Raw Images: {dataset_metadata['image_count']}")
print(f"Total Bounding Box Annotations: {dataset_metadata['annotation_count']}")

## 3. Auditable Label Derivation & Quality Gate

In [ ]:
label_audit_df = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "label_audit.csv")
print(f"Total Processed Worker Candidates: {len(label_audit_df)}")
print(label_audit_df["derived_label"].value_counts())

label_audit_df.head(5)

## 4. Leakage Prevention & Grouped Split Manifest

In [ ]:
split_manifest_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "split_manifest.csv")
print("Class Distribution across Grouped Splits:")
pd.crosstab(split_manifest_df["split"], split_manifest_df["class"], margins=True)

## 5. Model Training & Comparison

In [ ]:
with open(PROJECT_ROOT / "artifacts" / "models" / "metadata_resnet50.json", "r") as f:
    resnet_meta = json.load(f)
with open(PROJECT_ROOT / "artifacts" / "models" / "metadata_mobilenet_v3_large.json", "r") as f:
    mobilenet_meta = json.load(f)

print(f"ResNet50 Best Val Macro F1: {resnet_meta['validation_metrics']['macro_f1']:.4f}")
print(f"MobileNetV3 Best Val Macro F1: {mobilenet_meta['validation_metrics']['macro_f1']:.4f}")

## 6. Formal Model Selection & Test Set Benchmark

In [ ]:
with open(PROJECT_ROOT / "reports" / "evaluation_results.json", "r") as f:
    test_eval = json.load(f)

print(f"Champion Model: {test_eval['model_name']}")
print(f"Test Accuracy: {test_eval['metrics']['accuracy'] * 100:.2f}%")
print(f"Test Macro F1: {test_eval['metrics']['macro_f1']:.4f}")
print(f"Unsafe Class Recall: {test_eval['metrics']['safety_audit']['unsafe_recall_min']:.4f}")
print(f"Inference Latency: {test_eval['metrics']['latency_per_sample_ms']:.2f} ms/image")

## 7. Grad-CAM Explainability Overlay

In [ ]:
from app.predictor import get_predictor

predictor = get_predictor()
sample_crop_path = sorted(list((PROJECT_ROOT / "data" / "interim" / "crops").glob("*.jpg")))[0]

with Image.open(sample_crop_path) as img:
    sample_img = img.convert("RGB")

res, overlay = predictor.predict_with_gradcam(sample_img)
print(f"Prediction: {res['prediction']} ({res['confidence'] * 100:.1f}% confidence)")
print(f"Probabilities: {res['probabilities']}")

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(sample_img)
ax[0].set_title("Original Crop")
ax[0].axis("off")

ax[1].imshow(overlay)
ax[1].set_title(f"Grad-CAM: {res['prediction']}")
ax[1].axis("off")
plt.show()

## 8. Lineage & Checksums

In [ ]:
with open(PROJECT_ROOT / "reports" / "lineage.json", "r") as f:
    lineage = json.load(f)
print(json.dumps(lineage, indent=2))